In [13]:
def detect_text(path):
    """Detects text in the file."""
    from google.cloud import vision
    import os

    os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = 'sylvan-epoch-297907-3cca91bfa800.json'
    client = vision.ImageAnnotatorClient()

    with open(path, "rb") as image_file:
        content = image_file.read()

    image = vision.Image(content=content)

    response = client.text_detection(image=image)
    texts = response.text_annotations
    print("Texts:")

    for text in texts:
        print(f'\n"{text.description}"')

        vertices = [
            f"({vertex.x},{vertex.y})" for vertex in text.bounding_poly.vertices
        ]

        print("bounds: {}".format(",".join(vertices)))

    if response.error.message:
        raise Exception(
            "{}\nFor more info on error messages, check: "
            "https://cloud.google.com/apis/design/errors".format(response.error.message)
        )


In [14]:
detect_text('1758874980018.jpg')

Texts:

"OK
Ock
თოდუას კლინიკა
A
@
2
+995 322 57 57 57
info@toduaclinic.ge
www.toduaclinic.ge
სამედიცინო დოკუმენტაცია ფორმა № IV-100/ა
ც ნ ო ბ ა : FRM2025/09/642951
ჯანმრთელობის მდგომარეობის შესახებ
1. ცნობის გამცემი დაწესებულების დასახელება და მისამართი: შპს თოდუას კლინიკა თბილისი, თევდორე
მღვდლის N13
2. დაწესებულების დასახელება, მისამართი სადაც იგზავნება ცნობა:
საჭიროებისამებრ
3. პაციენტის სახელი და გვარი : ბულისკერია დავით
4 დაბადების თარიღი (რიცხვი/თვე/წელი) : 19/07/1993
ასაკი : 32 წლის
5. პირადი ნომერი: 62002005238
6. მისამართი: გალი ს. ოქუმი
მობილურის ნომერი: 533 33-43-38
7. სამუშაო ადგილი და თანამდებობა, (მოსწავლის/სტუდენტის შემთხვევაში - იმ სასწავლო
დაწესებულების/სკოლის დასახელება და კლასი/კურსი, სადაც იგი სწავლობს)
დასაქმებული
8. თარიღები:
ა) ამბულატორიაში მიმართვის : 2025-09-25
9. დასკვნა ჯანმრთელობის მდგომარეობის შესახებ ან სრული დიაგნოზი (ძირითადი დაავადება, თანმხლები
დაავადებები, გართულებები) ICD/ICD/
ძირითადი დაავადება/ICD/
M50.9 - კისრის სეგმენტის მალთაშუა დისკების დაზია

In [15]:
def detect_text(path, output_docx='extracted_text.docx'):
    """Detects text in the file and saves to docx."""
    import os
    from google.cloud import vision
    from docx import Document
    
    # Set credentials
    os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = 'account-key.json'
    
    client = vision.ImageAnnotatorClient()
    
    with open(path, "rb") as image_file:
        content = image_file.read()
    
    image = vision.Image(content=content)
    response = client.text_detection(image=image)
    texts = response.text_annotations
    
    if response.error.message:
        raise Exception(
            "{}\nFor more info on error messages, check: "
            "https://cloud.google.com/apis/design/errors".format(response.error.message)
        )
    
    # Create Word document
    doc = Document()
    doc.add_heading('Extracted Text from Image', 0)
    
    if texts:
        # First annotation contains all text
        full_text = texts[0].description
        doc.add_paragraph(full_text)
        
        print("Texts:")
        print(f'\n"{full_text}"')
        
        # Optionally add individual text elements with bounds
        doc.add_page_break()
        doc.add_heading('Detailed Text Elements', 1)
        
        for i, text in enumerate(texts[1:], 1):  # Skip first (it's the full text)
            doc.add_paragraph(f"Element {i}: {text.description}", style='List Number')
            vertices = [
                f"({vertex.x},{vertex.y})" for vertex in text.bounding_poly.vertices
            ]
            doc.add_paragraph(f"Bounds: {', '.join(vertices)}", style='List Bullet')
    else:
        doc.add_paragraph("No text detected in the image.")
    
    # Save document
    doc.save(output_docx)
    print(f"\nText saved to {output_docx}")
    
    return texts

# Usage
detect_text('1758874980018.jpg', 'test1.docx')

ModuleNotFoundError: No module named 'docx'